In [3]:
# !uv pip install xgboost lightgbm catboost

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, ExtraTreesRegressor
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# Try importing optional boosters
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

In [2]:
# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = os.path.join('..', '..', 'Dataset', 'DemandForecastingDataset', 'preprocess_demand_forecasting.csv') 
MODEL_PATH = os.path.join('..', '..', 'Model', 'DemandForecastingModel', 'demand_forecasting_models.pkl') 

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

Loaded: 76,000 rows × 22 cols


In [3]:
# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
# Parse date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Time features
df["DayOfYear"] = df["Date"].dt.dayofyear
df["Week"]      = df["Date"].dt.isocalendar().week.astype("Int64")
df["IsWeekend"] = df["Weekday"].isin(["Saturday", "Sunday"]).astype(int)
df["Quarter"]   = df["Date"].dt.quarter

# Cyclical encoding for Month & DayOfYear (seasonality is periodic)
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
df["Doy_sin"]   = np.sin(2 * np.pi * df["DayOfYear"] / 365)
df["Doy_cos"]   = np.cos(2 * np.pi * df["DayOfYear"] / 365)

# Interaction / ratio features
df["Price_Ratio_Comp"] = df["Price"] / (df["Competitor Pricing"] + 1e-6)
df["Discount_Amount"]  = df["Price"] * df["Discount"] / 100.0
df["Stock_Coverage"]   = df["Inventory Level"] / (df["Units Ordered"] + 1)

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Week,IsWeekend,Quarter,Month_sin,Month_cos,Doy_sin,Doy_cos,Price_Ratio_Comp,Discount_Amount,Stock_Coverage
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,52,1,1,0.5,0.866025,0.017213,0.999852,0.848244,3.636,0.770751
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,...,52,1,1,0.5,0.866025,0.017213,0.999852,0.871115,12.024,0.468000
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,...,52,1,1,0.5,0.866025,0.017213,0.999852,1.047603,6.294,0.402936
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,...,52,1,1,0.5,0.866025,0.017213,0.999852,1.028642,8.763,1.349515
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,...,52,1,1,0.5,0.866025,0.017213,0.999852,1.053845,0.000,0.558824


In [4]:
# ============================================================
# 3. SELECT FEATURES (drop leakage + IDs + raw date)
# ============================================================
# Excluded:
#   Units Sold          -> leakage (≈ Demand)
#   Sell_Through_Rate   -> derived from Units Sold
#   Discount_Price      -> redundant (Price × (1-Discount/100))
#   Store ID, Product ID-> high-cardinality IDs, handled via target encoding later if needed
#   Epidemic            -> constant zero
TARGET = "Demand"

DROP_COLS = [
    TARGET,
    "Date",
    "Units Sold",
    "Sell_Through_Rate",
    "Discount_Price",
    "Epidemic",
]

# Keep IDs out of default model but preserve option to encode
ID_COLS = ["Store ID", "Product ID"]

feature_cols = [c for c in df.columns if c not in DROP_COLS + ID_COLS + ["Weekday"]]
# Weekday handled as categorical below

# Add back Weekday as categorical
cat_cols = ["Category", "Region", "Weather Condition",
            "Seasonality", "Weekday"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"\nNumeric features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

X = df[num_cols + cat_cols].copy()
y = df[TARGET].copy()

# Drop rows with NaN target
mask = y.notna()
X, y = X[mask], y[mask]

print(f"\nFinal modeling set: {X.shape[0]:,} rows")


Numeric features (20): ['Inventory Level', 'Units Ordered', 'Price', 'Discount', 'Promotion', 'Competitor Pricing', 'Year', 'Month', 'Day', 'DayOfYear', 'Week', 'IsWeekend', 'Quarter', 'Month_sin', 'Month_cos', 'Doy_sin', 'Doy_cos', 'Price_Ratio_Comp', 'Discount_Amount', 'Stock_Coverage']
Categorical features (5): ['Category', 'Region', 'Weather Condition', 'Seasonality', 'Weekday']

Final modeling set: 76,000 rows


In [5]:
X.head()

,Inventory Level,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Year,Month,Day,DayOfYear,...,Doy_sin,Doy_cos,Price_Ratio_Comp,Discount_Amount,Stock_Coverage,Category,Region,Weather Condition,Seasonality,Weekday
0,195,252,72.72,5,0,85.73,2022,1,1,1,...,0.017213,0.999852,0.848244,3.636,0.770751,Electronics,North,Snowy,Winter,Saturday
1,117,249,80.16,15,1,92.02,2022,1,1,1,...,0.017213,0.999852,0.871115,12.024,0.468000,Clothing,North,Snowy,Winter,Saturday
2,247,612,62.94,10,1,60.08,2022,1,1,1,...,0.017213,0.999852,1.047603,6.294,0.402936,Clothing,North,Snowy,Winter,Saturday
3,139,102,87.63,10,0,85.19,2022,1,1,1,...,0.017213,0.999852,1.028642,8.763,1.349515,Electronics,North,Snowy,Winter,Saturday
4,152,271,54.41,0,0,51.63,2022,1,1,1,...,0.017213,0.999852,1.053845,0.000,0.558824,Groceries,North,Snowy,Winter,Saturday


In [6]:
y.head()

0    115
1    229
2    157
3     52
4     59
Name: Demand, dtype: int64

In [7]:
# ============================================================
# 4. TRAIN / TEST SPLIT (time-aware where possible)
# ============================================================
# Use random split here; swap to time-based split if Date matters
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
# ============================================================
# 5. PREPROCESSING PIPELINE
# ============================================================
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

In [9]:
# ============================================================
# 6. DEFINE MODELS
# ============================================================
models = {
    "LinearRegression": LinearRegression(),
    "Ridge":            Ridge(alpha=1.0),
    "Lasso":            Lasso(alpha=0.1, max_iter=5000),
    "ElasticNet":       ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
    "RandomForest":     RandomForestRegressor(
                            n_estimators=300, max_depth=None,
                            n_jobs=-1, random_state=42),
    "ExtraTrees":       ExtraTreesRegressor(
                            n_estimators=300, n_jobs=-1, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(
                            n_estimators=300, learning_rate=0.05,
                            max_depth=5, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(
                            max_iter=500, learning_rate=0.05,
                            random_state=42),
}

if HAS_XGB:
    models["XGBoost"] = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.9, colsample_bytree=0.9,
        n_jobs=-1, random_state=42, verbosity=0
    )

if HAS_LGBM:
    models["LightGBM"] = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        n_jobs=-1, random_state=42, verbosity=-1
    )

if HAS_CAT:
    models["CatBoost"] = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        verbose=0, random_state=42
    )


In [10]:
# ============================================================
# 7. TRAIN, EVALUATE, TRACK
# ============================================================
def evaluate(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE_%": mape}


results = {}
trained = {}

print("\n" + "=" * 70)
print("TRAINING MODELS")
print("=" * 70)

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    metrics = evaluate(y_test, y_pred)

    results[name] = metrics
    trained[name] = pipe

    print(f"{name:<22} | "
          f"R²={metrics['R2']:.4f} | "
          f"RMSE={metrics['RMSE']:.2f} | "
          f"MAE={metrics['MAE']:.2f} | "
          f"MAPE={metrics['MAPE_%']:.2f}%")



TRAINING MODELS
LinearRegression       | R²=0.5079 | RMSE=32.97 | MAE=25.87 | MAPE=38.01%
Ridge                  | R²=0.5078 | RMSE=32.97 | MAE=25.88 | MAPE=38.02%
Lasso                  | R²=0.5060 | RMSE=33.03 | MAE=25.93 | MAPE=38.34%
ElasticNet             | R²=0.4822 | RMSE=33.82 | MAE=26.61 | MAPE=40.21%
RandomForest           | R²=0.7024 | RMSE=25.64 | MAE=18.52 | MAPE=25.60%
ExtraTrees             | R²=0.7021 | RMSE=25.65 | MAE=18.38 | MAPE=25.56%
GradientBoosting       | R²=0.6866 | RMSE=26.31 | MAE=19.87 | MAPE=28.11%
HistGradientBoosting   | R²=0.7229 | RMSE=24.74 | MAE=18.42 | MAPE=25.76%
XGBoost                | R²=0.7236 | RMSE=24.71 | MAE=18.40 | MAPE=25.84%
LightGBM               | R²=0.7378 | RMSE=24.06 | MAE=17.74 | MAPE=24.80%
CatBoost               | R²=0.7041 | RMSE=25.56 | MAE=19.30 | MAPE=27.09%


In [11]:
# ============================================================
# 8. RANK & PICK BEST
# ============================================================
results_df = pd.DataFrame(results).T.sort_values("R2", ascending=False)
print("\n" + "=" * 70)
print("LEADERBOARD")
print("=" * 70)
print(results_df.round(4))

best_name = results_df.index[0]
best_model = trained[best_name]
print(f"\n Best model: {best_name}  (R² = {results_df.loc[best_name, 'R2']:.4f})")



LEADERBOARD
                          MAE     RMSE      R2   MAPE_%
LightGBM              17.7426  24.0640  0.7378  24.7979
XGBoost               18.4021  24.7063  0.7236  25.8438
HistGradientBoosting  18.4180  24.7375  0.7229  25.7636
CatBoost              19.3032  25.5623  0.7041  27.0876
RandomForest          18.5212  25.6362  0.7024  25.5994
ExtraTrees            18.3786  25.6497  0.7021  25.5587
GradientBoosting      19.8674  26.3092  0.6866  28.1054
LinearRegression      25.8716  32.9666  0.5079  38.0124
Ridge                 25.8753  32.9690  0.5078  38.0183
Lasso                 25.9266  33.0311  0.5060  38.3366
ElasticNet            26.6060  33.8174  0.4822  40.2130

🏆 Best model: LightGBM  (R² = 0.7378)


In [12]:
# ============================================================
# 9. FEATURE IMPORTANCE (for tree-based best model)
# ============================================================
try:
    # Get feature names after one-hot encoding
    prep = best_model.named_steps["prep"]
    ohe = prep.named_transformers_["cat"].named_steps["onehot"]
    cat_names = list(ohe.get_feature_names_out(cat_cols))
    all_names = num_cols + cat_names

    inner = best_model.named_steps["model"]
    if hasattr(inner, "feature_importances_"):
        imp = pd.Series(inner.feature_importances_, index=all_names)
        imp = imp.sort_values(ascending=False).head(15)
        print("\nTop 15 Feature Importances:")
        print(imp.round(4))
except Exception as e:
    print(f"(Feature importance skipped: {e})")



Top 15 Feature Importances:
Price                 4395
Inventory Level       3028
Units Ordered         2331
Discount_Amount       2268
Stock_Coverage        2241
Competitor Pricing    2134
Price_Ratio_Comp      1841
Doy_sin               1392
DayOfYear             1238
Doy_cos               1184
Day                   1133
Year                   711
Category_Groceries     647
Category_Clothing      640
Region_North           607
dtype: int32


In [13]:
# ============================================================
# 10. SAVE ALL MODELS TO PICKLE
# ============================================================
artifact = {
    "models": trained,          # dict of {name: fitted Pipeline}
    "results": results_df,      # leaderboard
    "best_model_name": best_name,
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "feature_columns": num_cols + cat_cols,
    "target": TARGET,
    "sklearn_version": __import__("sklearn").__version__,
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(artifact, f)

print(f"\n Saved {len(trained)} models + metadata → {MODEL_PATH}")


 Saved 11 models + metadata → ..\..\Model\DemandForecastingModel\demand_forecasting_models.pkl


In [16]:
# Load & Use Saved Models
import pickle
import pandas as pd

with open(MODEL_PATH, "rb") as f:
    art = pickle.load(f)

print(art["results"])                       # leaderboard
best = art["models"][art["best_model_name"]]  # best pipeline

# # Predict on new data
# new_df = pd.read_csv("new_data.csv")
# preds = best.predict(new_df[art["feature_columns"]])
# print(preds[:10])

                            MAE       RMSE        R2     MAPE_%
LightGBM              17.742615  24.063960  0.737799  24.797870
XGBoost               18.402069  24.706258  0.723616  25.843841
HistGradientBoosting  18.418029  24.737476  0.722917  25.763585
CatBoost              19.303193  25.562269  0.704132  27.087604
RandomForest          18.521214  25.636205  0.702418  25.599430
ExtraTrees            18.378555  25.649666  0.702105  25.558663
GradientBoosting      19.867440  26.309246  0.686588  28.105431
LinearRegression      25.871628  32.966591  0.507907  38.012358
Ridge                 25.875316  32.968963  0.507836  38.018261
Lasso                 25.926649  33.031085  0.505979  38.336586
ElasticNet            26.605986  33.817448  0.482177  40.213039


In [14]:
num_cols + cat_cols

['Inventory Level',
 'Units Ordered',
 'Price',
 'Discount',
 'Promotion',
 'Competitor Pricing',
 'Year',
 'Month',
 'Day',
 'DayOfYear',
 'Week',
 'IsWeekend',
 'Quarter',
 'Month_sin',
 'Month_cos',
 'Doy_sin',
 'Doy_cos',
 'Price_Ratio_Comp',
 'Discount_Amount',
 'Stock_Coverage',
 'Category',
 'Region',
 'Weather Condition',
 'Seasonality',
 'Weekday']